# Top-1,000 catalogue data-quality review

**Status as of 5 September 2026: structurally reproducible; needs eligibility and coverage revisions for bands-only or comprehensive city-scene claims.** The catalogue contains 1,000 unique Spotify artists and Wikidata entities and reconciles with its downstream files. Its current selection rules reproduce, but five verified solo projects still pass the entity-type gate.

Nonblank origin labels cover 74.9% of acts and 94.8% of summed captured monthly-listener counts. Strict FUA mapping covers 66.3% of acts and 90.7% of those counts. The dataset remains a frozen Spotify ranking within one archived Wikidata candidate frame, not a census of every UK band. See the [full dataset review](top1000_full_review_20260905.ipynb) for the current findings.

## Scope and definitions

This audit treats `popularity_first_top1000_20260718T204522Z_bands.csv` as the canonical catalogue. It checks integrity, selection gates, origin coverage, FUA mapping, and the interactive derivative. The population is not all UK bands: it is the set of Spotify IDs returned for entities in one archived Wikidata response, after metric capture, identity review, an eligibility rule, redirect deduplication, and ranking by Spotify monthly listeners.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'requirements.txt').exists() and (p / 'data').exists())
catalog = pd.read_csv(root / 'data/processed/popularity_first_top1000_20260718T204522Z_bands.csv', keep_default_na=False)
candidates = pd.read_csv(root / 'data/interim/uk_group_spotify_candidates_20260718T201100Z.csv', keep_default_na=False)
metrics = pd.read_csv(root / 'data/processed/uk_group_spotify_metrics_20260718T204522Z.csv', keep_default_na=False)
identity = pd.read_csv(root / 'data/interim/popularity_first_top1000_20260718T204522Z_identity_audit.csv', keep_default_na=False)
fua = pd.read_csv(root / 'data/interim/popularity_first_top1000_20260718T204522Z_fua_mapping_audit.csv', keep_default_na=False)
dashboard = json.loads((root / 'interactive/public/data/dashboard.json').read_text())
capture_report = json.loads((root / 'data/raw/spotify/uk_group_spotify_metrics_20260718T204522Z_report.json').read_text())
cutoff = int(catalog['monthly_listeners'].min())

## 1. Structural integrity

These assertions cover the catalogue's shape, keys, rank ordering, identifiers, and non-negative measures.

In [2]:
assert len(catalog) == 1_000
assert catalog['popularity_rank'].tolist() == list(range(1, 1_001))
assert catalog['returned_spotify_id'].is_unique
assert catalog['capture_key'].is_unique
assert catalog['wikidata_qid'].is_unique
assert catalog['monthly_listeners'].is_monotonic_decreasing
assert catalog[['monthly_listeners', 'followers']].ge(0).all().all()
assert catalog['returned_spotify_id'].str.fullmatch(r'[A-Za-z0-9]{22}').all()
assert catalog['wikidata_qid'].str.fullmatch(r'Q\d+').all()
pd.Series({
    'rows': len(catalog),
    'monthly_listener_total': int(catalog['monthly_listeners'].sum()),
    'follower_total': int(catalog['followers'].sum()),
    'listener_median': int(catalog['monthly_listeners'].median()),
    'cutoff_monthly_listeners': cutoff,
    'exact_duplicate_rows': int(catalog.duplicated().sum()),
})

rows                              1000
monthly_listener_total      2304280501
follower_total               927791496
listener_median                 295536
cutoff_monthly_listeners         29728
exact_duplicate_rows                 0
dtype: int64

## 2. Selection funnel and cutoff sensitivity

The candidate and metric tables reconcile. The tail remains sensitive: the selected rank-1,000 value is 381 monthly listeners above the next eligible row.

In [3]:
pool = identity[identity['identity_accepted'] & identity['band_eligible'] & ~identity['redirect_duplicate'] & ~identity['entity_duplicate']].sort_values('monthly_listeners', ascending=False)
assert pool.head(1_000)['returned_spotify_id'].tolist() == catalog['returned_spotify_id'].tolist()
near = pool[pool['monthly_listeners'].between(cutoff * 0.9, cutoff * 1.1)]
pd.Series({
    'wikidata_bindings': 1_928,
    'candidate_spotify_ids': len(candidates),
    'successful_metric_rows': len(metrics),
    'metric_failures': len(capture_report['metric_failures']),
    'accepted_eligible_unique_pool': len(pool),
    'selected_rows': len(catalog),
    'next_eligible_monthly_listeners': int(pool.iloc[1_000]['monthly_listeners']),
    'cutoff_gap': cutoff - int(pool.iloc[1_000]['monthly_listeners']),
    'rows_within_10pct_of_cutoff': len(near),
})

wikidata_bindings                   1928
candidate_spotify_ids               1775
successful_metric_rows              1749
metric_failures                       26
accepted_eligible_unique_pool       1633
selected_rows                       1000
next_eligible_monthly_listeners    29347
cutoff_gap                           381
rows_within_10pct_of_cutoff           53
dtype: int64

## 3. Selection-gate review

Eligibility uses entity type, so groups such as Electric Light Orchestra and The Cinematic Orchestra remain eligible. Name normalization covers leading articles and `+`/`&`/`and` variants, reviewed aliases are explicit, and duplicate Wikidata entities are removed after popularity sorting. The rejection table below contains ten capture rows representing eight distinct artist names: six orchestras, Goreshit and Olly Alexander (Years & Years). Redirect duplicates account for the difference between rows and names.

These checks establish that the implemented rules reproduce, not that every inclusion is a band. The full review verifies five solo projects still included under the frozen “musical group” label: Sidewalks and Skeletons, IAMX, Cinnamon Chasers, Ex:Re and Hallucinogen. Their exclusion and replacement decisions remain pending. Historical groups now operating as solo acts also need a consistent eligibility rule.

In [4]:
accepted_orchestra_names = identity[identity['spotify_name'].isin(['Electric Light Orchestra', 'The Cinematic Orchestra', 'Hidden Orchestra'])][
    ['spotify_name', 'instance_label', 'monthly_listeners', 'eligibility_status', 'identity_status']
].sort_values('monthly_listeners', ascending=False)
reviewed_rejections = identity[identity['identity_decision'].eq('reject') & identity['monthly_listeners'].ge(cutoff)][
    ['spotify_name', 'band_name', 'instance_label', 'monthly_listeners', 'identity_status']
].sort_values('monthly_listeners', ascending=False)
unreviewed_eligible_mismatches = identity[identity['identity_status'].eq('rejected_name_mismatch') & identity['band_eligible'] & identity['monthly_listeners'].ge(cutoff)]
assert unreviewed_eligible_mismatches.empty
display(accepted_orchestra_names)
print(f'Explicit reviewed rejections at or above cutoff: {len(reviewed_rejections)}')
display(reviewed_rejections)

,spotify_name,instance_label,monthly_listeners,eligibility_status,identity_status
27,Electric Light Orchestra,musical group,18330664,eligible_group_or_duo,accepted_exact_name
104,The Cinematic Orchestra,musical group,5329785,eligible_group_or_duo,accepted_exact_name
725,Hidden Orchestra,musical group,127742,eligible_group_or_duo,accepted_exact_name


Explicit reviewed rejections at or above cutoff: 10


,spotify_name,band_name,instance_label,monthly_listeners,identity_status
33,Olly Alexander (Years & Years),Years & Years,musical group|solo musical project,15856316,rejected_reviewed
95,Royal Philharmonic Orchestra,Royal Philharmonic Orchestra,musical group,5838561,rejected_reviewed
96,Royal Philharmonic Orchestra,Royal Philharmonic Orchestra,musical group,5838561,rejected_reviewed
175,London Philharmonic Orchestra,London Philharmonic Orchestra,musical group,2930933,rejected_reviewed
454,Goreshit,Q113361759,musical group,402546,rejected_reviewed
685,London Session Orchestra,The London Session Orchestra,musical group,152953,rejected_reviewed
930,Mantovani & His Orchestra,Mantovani and His Orchestra,musical group,49525,rejected_reviewed
931,Mantovani & His Orchestra,Mantovani and His Orchestra,musical group,49525,rejected_reviewed
963,Geoff Love & His Orchestra,Geoff Love & His Orchestra,musical group,43394,rejected_reviewed
991,Orchestra of the Swan,Orchestra of the Swan,musical group,38155,rejected_reviewed


The seven non-band inclusions identified in the earlier review are absent from the current catalogue. The regression check below verifies that specific exclusion list; it does not cover the five additional solo-project inclusions identified in the full review.

In [5]:
clear_conflicts = ['Peppa Pig', 'Sea of Thieves', 'London Elektricity', 'The Yogscast', 'Sidemen', 'Noise Foundation', 'Man-Made Sunshine']
conflicts = catalog[catalog['spotify_name'].isin(clear_conflicts)][
    ['popularity_rank', 'spotify_name', 'instance_label', 'monthly_listeners', 'followers']
].sort_values('popularity_rank')
assert conflicts.empty
display(conflicts)
print('Combined monthly listeners:', f"{conflicts['monthly_listeners'].sum():,}")

,popularity_rank,spotify_name,instance_label,monthly_listeners,followers


Combined monthly listeners: 0


## 4. Origin and FUA coverage

Missing origin labels are concentrated in the long tail: all of the first 100 acts have labels, compared with 57 of ranks 801–900 and 61 of ranks 901–1,000. This matters more for band-count and scene-breadth comparisons than for audience-weighted totals.

The catalogue has 749 nonblank origin labels, covering 94.8% of summed monthly-listener counts; six labels identify only a nation or country. Strict FUA assignments cover 663 acts and 90.7% of listener counts. The `fua_mapped_bands` calculation below includes both strict and reviewed-extended assignments: 666 acts and 91.0% of listener counts. Neither label completeness nor mapping completeness establishes the truth of every formation claim.

The origin audit retains 330 proposed, source-variant, unresolved or contested records. Its 18 corrections and 11 resolutions already match the current catalogue.

In [6]:
catalog['decile'] = pd.cut(catalog['popularity_rank'], bins=range(0, 1_001, 100), labels=range(1, 11))
deciles = catalog.groupby('decile', observed=True).agg(
    bands=('spotify_name', 'size'),
    resolved_bands=('origin_cluster', lambda s: s.ne('').sum()),
    monthly_listeners=('monthly_listeners', 'sum'),
)
deciles['origin_coverage'] = deciles['resolved_bands'] / deciles['bands']
resolved = catalog['origin_cluster'].ne('')
mapped = fua['fua_code'].ne('')
display(deciles)
pd.Series({
    'origin_resolved_bands': int(resolved.sum()),
    'origin_band_coverage': float(resolved.mean()),
    'resolved_listener_share': float(catalog.loc[resolved, 'monthly_listeners'].sum() / catalog['monthly_listeners'].sum()),
    'fua_mapped_bands': int(mapped.sum()),
    'fua_mapped_listener_share': float(fua.loc[mapped, 'monthly_listeners'].sum() / catalog['monthly_listeners'].sum()),
})

,bands,resolved_bands,monthly_listeners,origin_coverage
decile,,,,
1,100,100,1636569832,1.00
2,100,86,358773600,0.86
3,100,84,143670036,0.84
4,100,78,67962154,0.78
5,100,75,37975404,0.75
6,100,69,24586884,0.69
7,100,72,15642077,0.72
8,100,67,9348695,0.67
9,100,57,5985624,0.57


origin_resolved_bands        749.000000
origin_band_coverage           0.749000
resolved_listener_share        0.947937
fua_mapped_bands             666.000000
fua_mapped_listener_share      0.909593
dtype: float64

## 5. Downstream reconciliation

The FUA audit and interactive dashboard reproduce the canonical IDs, ranks, names, listener counts, and follower counts. Blank CSV strings are normalized to JSON nulls for the origin comparison.

In [7]:
assert fua['returned_spotify_id'].tolist() == catalog['returned_spotify_id'].tolist()
assert fua['monthly_listeners'].tolist() == catalog['monthly_listeners'].tolist()
dashboard_bands = pd.DataFrame(dashboard['bands']).sort_values('catalogRank').reset_index(drop=True)
dashboard_columns = {'catalogRank': 'popularity_rank', 'name': 'spotify_name', 'monthlyListeners': 'monthly_listeners', 'followers': 'followers'}
for dashboard_column, catalog_column in dashboard_columns.items():
    assert dashboard_bands[dashboard_column].tolist() == catalog[catalog_column].tolist()
assert dashboard_bands['id'].tolist() == catalog['returned_spotify_id'].tolist()
assert dashboard_bands['originCluster'].fillna('').tolist() == catalog['origin_cluster'].tolist()
print('Canonical catalogue, FUA audit, and dashboard reconcile: PASS')

Canonical catalogue, FUA audit, and dashboard reconcile: PASS


## 6. Conclusions and required fixes

1. **The implemented selection rules reproduce; eligibility still needs revision.** Five verified solo projects pass the type-based gate. Adjudicate them through the existing override table, review replacement identities and define how former groups/current solo acts are treated.
2. **Downstream reconciliation passes.** The catalogue, FUA audit and interactive dashboard reproduce the same 1,000 ranked artists and measures. Agreement between derivatives does not independently validate artist identity or formation history.
3. **Keep the scope precise.** Describe this as a frozen Spotify-ranked catalogue drawn from an archived Wikidata candidate frame, not the definitive top 1,000 UK bands. The 26 metric failures also have unknown captured reach rather than demonstrated below-cutoff values.
4. **Preserve the provenance limitation.** The raw Wikidata response does not archive the exact SPARQL text and query URL/hash, so candidate-frame exhaustiveness cannot be reconstructed from that file alone. Label Spotify values with their 18 July 2026 snapshot date.
5. **Separate origin labels, FUA assignments and audit status.** Label coverage is 74.9% by act and 94.8% by listener counts; strict FUA coverage is 66.3% and 90.7%. Extended mapping covers 66.6% and 91.0%. There are 330 pending or contested origin records; all 29 recorded corrections/resolutions are already applied.
6. **Qualify city-scene claims.** Of 59 FUAs with mapped acts, 25 have one selected act; 24 of the 83 population FUAs have none. These are catalogue counts, not evidence of absent musical activity. Report counts, coverage and largest-band sensitivity alongside normalized reach.